### FACE NORMAL BILATERAL FILTERING

In [2]:
import open3d as o3d
import numpy as np

mesh = o3d.geometry.TriangleMesh.create_sphere(
    radius=1.0,
    resolution=10
)

mesh.compute_triangle_normals()

vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
face_normals = np.asarray(mesh.triangle_normals)

print("Vertices:", len(vertices))
print("Faces:", len(triangles))

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Vertices: 182
Faces: 360


In [3]:
face_neighbors = [[] for _ in range(len(triangles))]

for i in range(len(triangles)):
    for j in range(i + 1, len(triangles)):

        shared = set(triangles[i]) & set(triangles[j])

        if len(shared) == 2:
            face_neighbors[i].append(j)
            face_neighbors[j].append(i)

In [4]:
face_id = 10

print("Face:", face_id)
print("Normal:", face_normals[face_id])
print("Neighbors:", face_neighbors[face_id])
n_i = face_normals[face_id]

for neighbor in face_neighbors[face_id]:

    n_j = face_normals[neighbor]

    similarity = np.dot(n_i, n_j)

    print(
        "Neighbor:", neighbor,
        "Normal similarity:", similarity
    )

Face: 10
Normal: [-0.02476918  0.15638647  0.98738531]
Neighbors: [8, 12, 50]
Neighbor: 8 Normal similarity: 0.9987729750882129
Neighbor: 12 Normal similarity: 0.9987729750882129
Neighbor: 50 Normal similarity: 0.9500927762787201


In [5]:
centroids = np.zeros((len(triangles), 3))

for i, triangle in enumerate(triangles):
    centroids[i] = vertices[triangle].mean(axis=0)

In [6]:
sigma_s = 0.2
sigma_n = 0.2

face_id = 10

n_i = face_normals[face_id]
c_i = centroids[face_id]

for neighbor in face_neighbors[face_id]:

    n_j = face_normals[neighbor]
    c_j = centroids[neighbor]

    # Spatial difference
    distance = np.linalg.norm(c_i - c_j)

    # Normal difference
    similarity = np.dot(n_i, n_j)

    # Spatial weight
    w_s = np.exp(
        -(distance ** 2) / (2 * sigma_s ** 2)
    )

    # Normal weight
    w_n = np.exp(
        -((1 - similarity) ** 2) / (2 * sigma_n ** 2)
    )

    # Bilateral weight
    w = w_s * w_n

    print(
        f"Face {neighbor}: "
        f"distance={distance:.4f}, "
        f"similarity={similarity:.4f}, "
        f"weight={w:.4f}"
    )

Face 8: distance=0.0637, similarity=0.9988, weight=0.9506
Face 12: distance=0.0637, similarity=0.9988, weight=0.9506
Face 50: distance=0.2060, similarity=0.9501, weight=0.5703


In [8]:
filtered_normals = face_normals.copy()

for i in range(len(triangles)):

    n_i = face_normals[i]
    c_i = centroids[i]

    weighted_normal = np.zeros(3)
    weight_sum = 0.0

    for j in face_neighbors[i]:

        n_j = face_normals[j]
        c_j = centroids[j]

        distance = np.linalg.norm(c_i - c_j)
        similarity = np.dot(n_i, n_j)

        w_s = np.exp(
            -(distance ** 2) / (2 * sigma_s ** 2)
        )

        w_n = np.exp(
            -((1 - similarity) ** 2) / (2 * sigma_n ** 2)
        )

        w = w_s * w_n

        weighted_normal += w * n_j
        weight_sum += w

    if weight_sum > 0:
        filtered_normals[i] = weighted_normal / weight_sum

    # Normalize
    filtered_normals[i] /= np.linalg.norm(filtered_normals[i])

In [9]:
noisy_mesh = o3d.geometry.TriangleMesh(mesh)

vertices = np.asarray(noisy_mesh.vertices)

noise_strength = 0.05

noise = np.random.normal(
    0,
    noise_strength,
    vertices.shape
)

vertices += noise

noisy_mesh.compute_triangle_normals()

noisy_vertices = np.asarray(noisy_mesh.vertices)
noisy_normals = np.asarray(noisy_mesh.triangle_normals)

In [10]:
o3d.visualization.draw_plotly(
    [mesh],
    window_name="Clean Mesh"
)



In [11]:
o3d.visualization.draw_plotly(
    [noisy_mesh],
    window_name="Noisy Mesh"
)

In [12]:
clean_normals = np.asarray(mesh.triangle_normals)

dot = np.sum(
    clean_normals * noisy_normals,
    axis=1
)

dot = np.clip(dot, -1.0, 1.0)

angles = np.arccos(dot)

mean_error = np.mean(angles)

print(
    "Mean normal error:",
    np.degrees(mean_error),
    "degrees"
)

Mean normal error: 24.777447249522687 degrees


In [13]:
face_normals = noisy_normals

filtered_normals = face_normals.copy()

sigma_s = 0.2
sigma_n = 0.2

for i in range(len(triangles)):

    n_i = face_normals[i]
    c_i = centroids[i]

    weighted_normal = np.zeros(3)
    weight_sum = 0.0

    for j in face_neighbors[i]:

        n_j = face_normals[j]
        c_j = centroids[j]

        distance = np.linalg.norm(c_i - c_j)

        similarity = np.dot(n_i, n_j)
        similarity = np.clip(similarity, -1.0, 1.0)

        w_s = np.exp(
            -(distance ** 2) / (2 * sigma_s ** 2)
        )

        w_n = np.exp(
            -((1 - similarity) ** 2) /
            (2 * sigma_n ** 2)
        )

        w = w_s * w_n

        weighted_normal += w * n_j
        weight_sum += w

    if weight_sum > 0:
        filtered_normals[i] = (
            weighted_normal / weight_sum
        )

        filtered_normals[i] /= np.linalg.norm(
            filtered_normals[i]
        )